# Explore a mapcv dataset

This notebook opens a dataset written by `mapcv generate` and answers the first questions you
have before training: what is in it, how the splits are laid out, how balanced the classes are,
and whether the masks line up with the imagery.

It only reads files, so it works on any mapcv output. By default it points at the
[quickstart](../quickstart/) dataset. Create that first (about 30 seconds):

```bash
cd examples/quickstart
mapcv generate mapcv.yaml
```

Requirements: `mapcv` and `matplotlib` (`pip install mapcv matplotlib`).

> The imagery comes from Esri World Imagery and the labels from OpenStreetMap
> (© OpenStreetMap contributors, ODbL). Don't commit this notebook with its outputs: they
> embed the imagery.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Point this at any mapcv dataset directory (writer.staging_dir).
DATASET = Path("../quickstart/dataset")

if not (DATASET / "manifest.json").exists():
    raise FileNotFoundError(
        f"No dataset at {DATASET.resolve()}. Run `mapcv generate mapcv.yaml` in "
        "examples/quickstart first, or change DATASET."
    )

## 1. The manifest

`manifest.json` describes the whole dataset: where the imagery came from, the band order,
the patch shape, the coordinate reference system and the class map. It also lists every patch
with its raster position (`row`, `col`) and its pixel count per class, so many questions can
be answered without opening a single image.

In [ ]:
manifest = json.loads((DATASET / "manifest.json").read_text())
patches = manifest["patches"]

for key in ("version", "source_type", "product_id", "bands", "dtype", "patch_shape", "crs"):
    print(f"{key:12} {manifest[key]}")
print(f"{'class_map':12} {manifest['class_map']}")
print(f"{'patches':12} {len(patches)}")
print()
print(json.dumps(patches[0], indent=2))

Class `0` is always background (no polygon). The other IDs come from `class_map`; a
dataset generated without `labels.label_field` has an empty class map and uses `1` for every
polygon.

In [ ]:
class_names = {0: "background"}
class_names.update({cid: name for name, cid in manifest["class_map"].items()})
if len(class_names) == 1 and any(p["mask_filename"] for p in patches):
    class_names[1] = "foreground"
print(class_names)

## 2. The splits

`splits/train.txt`, `val.txt` and `test.txt` list one image filename per line. The
`10/`, `20/`, `30/` folders split `train` further into labeled and unlabeled lists for
semi-supervised experiments.

In [ ]:
def read_split(name):
    path = DATASET / "splits" / f"{name}.txt"
    return path.read_text().split() if path.exists() else []


splits = {name: read_split(name) for name in ("train", "val", "test")}
split_of = {filename: name for name, files in splits.items() for filename in files}
unused = len(patches) - len(split_of)

for name, files in splits.items():
    print(f"{name:>5}: {len(files):4d} patches ({len(files) / len(patches):.0%})")
print(f"not in any split: {unused} (patches overlapping a held-out patch are left out)")

for ratio_dir in sorted(p for p in (DATASET / "splits").iterdir() if p.is_dir()):
    labeled = (ratio_dir / "labeled.txt").read_text().split()
    print(f"{ratio_dir.name:>4}% labeled subset of train: {len(labeled)} patches")

The default `spatial` strategy assigns whole blocks of the raster to one split, so
neighbouring (or overlapping) patches never end up on both sides of the train/test line. Plot
each patch at its raster position to see the blocks.

In [ ]:
colors = {"train": "tab:blue", "val": "tab:orange", "test": "tab:red", None: "lightgray"}
size = manifest["sampler"]["patch_size"]

fig, ax = plt.subplots(figsize=(7, 5))
for patch in patches:
    split = split_of.get(patch["filename"])
    ax.add_patch(
        plt.Rectangle(
            (patch["col"], patch["row"]),
            size,
            size,
            facecolor=colors[split],
            edgecolor="white",
            linewidth=0.5,
        )
    )
ax.set_xlim(0, max(p["col"] for p in patches) + size)
ax.set_ylim(max(p["row"] for p in patches) + size, 0)
ax.set_aspect("equal")
ax.set_xlabel("column (px)")
ax.set_ylabel("row (px)")
ax.legend(
    handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()],
    labels=["train", "val", "test", "unused"],
    loc="upper left",
    bbox_to_anchor=(1, 1),
)
ax.set_title("Where each split's patches are")
plt.show()

## 3. Class balance

The per-patch pixel counts in the manifest add up to the class balance of each split.
Segmentation datasets are usually dominated by background; if a class is rare, consider a
weighted loss or `sampler.min_label_ratio`.

In [ ]:
def class_pixels(files):
    by_name = {p["filename"]: p for p in patches}
    totals = Counter()
    for filename in files:
        for cid, count in by_name[filename]["per_class_pixel_counts"].items():
            totals[int(cid)] += count
    return totals


ids = sorted(class_names)
fig, ax = plt.subplots(figsize=(7, 3.5))
width = 0.8 / len(splits)
for i, (name, files) in enumerate(splits.items()):
    totals = class_pixels(files)
    total = sum(totals.values()) or 1
    shares = [totals[cid] / total for cid in ids]
    ax.bar(np.arange(len(ids)) + i * width, shares, width, label=name, color=colors[name])
ax.set_xticks(np.arange(len(ids)) + width * (len(splits) - 1) / 2)
ax.set_xticklabels([class_names[cid] for cid in ids])
ax.set_ylabel("share of pixels")
ax.set_title("Class balance per split")
ax.legend()
plt.show()

## 4. Images and masks

Image patches are PNG/JPG for XYZ imagery and bands-first `float32` NPY for Sentinel-2. Masks
are single-channel PNGs whose pixel values are class IDs. The helper below turns either image
kind into something `imshow` can display (for NPY it picks red, green and blue and stretches
them).

In [ ]:
def load_rgb(filename):
    path = DATASET / "Images" / filename
    if path.suffix != ".npy":
        return np.asarray(Image.open(path).convert("RGB"))
    bands = manifest["bands"]
    array = np.load(path)  # (bands, height, width), float32, NaN = no data
    rgb_idx = [bands.index(b) for b in ("b04", "b03", "b02")] if "b04" in bands else [0, 0, 0]
    rgb = np.moveaxis(array[rgb_idx], 0, -1)
    lo, hi = np.nanpercentile(rgb, [2, 98])
    return np.nan_to_num(np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1))


def load_mask(filename):
    return np.asarray(Image.open(DATASET / "Masks" / filename))


palette = plt.get_cmap("tab10")


def overlay(rgb, mask, alpha=0.45):
    rgb = rgb.astype(float) / (255 if rgb.dtype == np.uint8 else 1)
    out = rgb.copy()
    for cid in np.unique(mask):
        if cid == 0:
            continue
        color = np.array(palette(int(cid) % 10)[:3])
        out[mask == cid] = (1 - alpha) * rgb[mask == cid] + alpha * color
    return out

A few random test patches: the image on top, the mask overlaid below. Check that
polygons sit on the right objects; a consistent shift usually means the labels and imagery
disagree about coordinates.

In [ ]:
rng = np.random.default_rng(0)
by_name = {p["filename"]: p for p in patches}
pool = splits["test"] or [p["filename"] for p in patches]
chosen = rng.choice(pool, size=min(6, len(pool)), replace=False)

fig, axes = plt.subplots(2, len(chosen), figsize=(2.6 * len(chosen), 5.4), squeeze=False)
for col, filename in enumerate(chosen):
    rgb = load_rgb(filename)
    mask_name = by_name[filename]["mask_filename"]
    axes[0, col].imshow(rgb)
    axes[0, col].set_title(filename.split(".")[0][-4:], fontsize=9)
    axes[1, col].imshow(overlay(rgb, load_mask(mask_name)) if mask_name else rgb)
for ax in axes.ravel():
    ax.axis("off")
handles = [plt.Rectangle((0, 0), 1, 1, color=palette(cid % 10)) for cid in ids if cid]
fig.legend(handles, [class_names[cid] for cid in ids if cid], loc="lower center", ncol=len(handles))
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

## Next

* Train a model on this dataset: [02-train-a-segmentation-model.ipynb](02-train-a-segmentation-model.ipynb).
* Re-split without regenerating: `mapcv split ../quickstart/dataset --test-ratio 0.3`.
* Load patches in your own code with [`scripts/torch_dataset.py`](../scripts/torch_dataset.py).
* Point `DATASET` at the [Sentinel-2 example](../sentinel2-landcover/) output to explore a
  multispectral, multi-class dataset with the same code.